# 02 - SQL Layer
The cleaned data is split into three normalised tables in SQLite (`customers`, `services`, `billing`, joined on `customerID`), then queried the way a business stakeholder would. The queries live in `sql/queries.sql`; this notebook runs them and records the takeaway for each.

**Design note:** the Telco data has no signup date, so a 'cohort retention by signup month' query is impossible without inventing dates. Q2 reports retention by *tenure bucket* instead.

In [1]:
import sys, re, sqlite3; sys.path.append('..')
import pandas as pd
from src import build_db
pd.set_option('display.width', 200)
build_db.build()
con = sqlite3.connect(build_db.DB)
text = open('../sql/queries.sql').read()
blocks = re.split(r'(?m)^-- (Q\d+): ', text)[1:]
Q = {tag: body.partition('\n') for tag, body in zip(blocks[::2], blocks[1::2])}
def q(tag):
    title, _, sql = Q[tag]; print(tag + ':', title); return pd.read_sql_query(sql, con)

customers: 7043 rows
services: 7043 rows
billing: 7043 rows


## Schema

In [2]:
pd.read_sql_query("SELECT name, sql FROM sqlite_master WHERE type='table'", con)

,name,sql
0,customers,"CREATE TABLE ""customers"" (\n""customerID"" TEXT,..."
1,services,"CREATE TABLE ""services"" (\n""customerID"" TEXT,\..."
2,billing,"CREATE TABLE ""billing"" (\n""customerID"" TEXT,\n..."


## Q1 - Churn rate by contract type

In [3]:
q('Q1')

Q1: Churn rate by contract type. Where is the problem concentrated?


,Contract,customers,churned,churn_rate_pct
0,Month-to-month,3875,1655,42.7
1,One year,1473,166,11.3
2,Two year,1695,48,2.8


**Takeaway:** month-to-month contracts churn at **42.7%** vs 11.3% (1-year) and 2.8% (2-year).

## Q2 - Retention by tenure bucket

In [4]:
q('Q2')

Q2: Retention by tenure bucket (replaces "cohort by signup month": the data has no signup date).


,tenure_bucket,customers,churn_rate_pct,retention_pct
0,1) 0-6 mo,1481,52.9,47.1
1,2) 7-12 mo,705,35.9,64.1
2,3) 13-24 mo,1024,28.7,71.3
3,4) 25-48 mo,1594,20.4,79.6
4,5) 49+ mo,2239,9.5,90.5


**Takeaway:** retention climbs from **47.1% in the first 6 months to 90.5% after 49 months**: the first year is the danger zone.

## Q3 - Tenure and bill by internet service and churn status

In [5]:
q('Q3')

Q3: Average tenure and monthly charge by internet service, split by churn status.


,InternetService,status,customers,avg_tenure_mo,avg_monthly_charge
0,DSL,churned,459,14.1,49.08
1,DSL,retained,1962,37.2,60.21
2,Fiber optic,churned,1297,20.2,88.13
3,Fiber optic,retained,1799,42.1,93.93
4,No,churned,113,8.2,20.37
5,No,retained,1413,32.3,21.14


**Takeaway:** within every internet type, churners have far shorter tenure (fiber: **20 vs 42 months**) and pay *slightly less*, not more. They have fewer add-on services, so price alone is not what drives them away. (Overall churners look pricier only because fiber, the most expensive product, is over-represented among them.)

## Q4 - Monthly revenue lost to churn, by contract (window function for share)

In [6]:
q('Q4')

Q4: Monthly revenue already lost to churn, by contract, and its share of total monthly revenue.


,Contract,total_mrr,lost_mrr,lost_pct_of_contract_mrr,share_of_all_lost_mrr_pct
0,Month-to-month,257294.0,120847.0,47.0,86.9
1,One year,95817.0,14118.0,14.7,10.1
2,Two year,103006.0,4165.0,4.0,3.0


**Takeaway:** month-to-month customers account for **86.9%** of all monthly revenue lost to churn ($120.8K of $139.1K).

## Q5 - Revenue at risk among current customers matching the high-risk profile

In [7]:
q('Q5')

Q5: Revenue at risk among CURRENT customers who match the high-risk profile found in Q1-Q3


,active_high_risk_customers,monthly_revenue_at_risk,annualised_revenue_at_risk
0,273,22005.0,264058.0


**Takeaway:** 273 active month-to-month, fiber, <=12-month customers represent about **$264K of annualised revenue**. This is a rule-based first cut; Phase 5 replaces it with model scores.

## Q6 - Payment methods ranked by churn within each contract (`RANK() OVER PARTITION BY`)

In [8]:
q('Q6')

Q6: Within each contract type, rank payment methods by churn rate (window function).


,Contract,PaymentMethod,customers,churn_rate_pct,risk_rank
0,Month-to-month,Electronic check,1850,53.7,1
1,Month-to-month,Bank transfer (automatic),589,34.1,2
2,Month-to-month,Credit card (automatic),543,32.8,3
3,Month-to-month,Mailed check,893,31.6,4
4,One year,Electronic check,347,18.4,1
5,One year,Credit card (automatic),398,10.3,2
6,One year,Bank transfer (automatic),391,9.7,3
7,One year,Mailed check,337,6.8,4
8,Two year,Electronic check,168,7.7,1
9,Two year,Bank transfer (automatic),564,3.4,2


**Takeaway:** electronic check is rank 1 (highest churn) within **every** contract type, at 53.7% for month-to-month customers vs ~32% for the other methods.

## Q7 - Churn by monthly-charge decile with cumulative churners captured (`NTILE`, running `SUM OVER`)

In [9]:
q('Q7')

Q7: Churn by monthly-charge decile (NTILE) with cumulative share of all churners captured,


,price_decile,min_charge,customers,churners,churn_rate_pct,cum_churners_captured_pct
0,1,102.60,705,174,24.7,9.3
1,2,94.25,705,290,41.1,24.8
2,3,85.50,705,240,34.0,37.7
3,4,79.10,704,268,38.1,52.0
4,5,70.35,704,266,37.8,66.2
5,6,58.75,704,152,21.6,74.4
6,7,45.85,704,174,24.7,83.7
7,8,25.05,704,177,25.1,93.2
8,9,20.05,704,67,9.5,96.7
9,10,18.25,704,61,8.7,100.0


**Takeaway:** churn is *not* monotonic in price: the top decile churns at 24.7%, below deciles 2-5 (34-41%), and the cheapest two deciles churn under 10%. Targeting by price alone would reach only 52% of churners with the top 40% of customers, a weak rule that motivates the model.

## Q8 - Protective add-ons vs churn

In [10]:
q('Q8')

Q8: Number of protective add-ons (security, backup, protection, tech support) vs churn.


,n_protective,customers,churn_rate_pct
0,0,1267,56.7
1,1,1467,38.9
2,2,1372,23.8
3,3,941,12.4
4,4,470,5.3


**Takeaway:** churn falls steadily from **56.7% with no protective add-ons to 5.3% with all four**.

In [11]:
con.close()